# LegalQA Main 04 — Adaptive fallback / truncated repair

Baseline: submission private **1.918 câu, 0.5713**. Mặc định `adaptive_dev`.
Sinh lại chọn lọc với output **2.048 → 3.072 → 4.096 token**, input tối đa 4.096,
có một lượt sửa prompt/context khi lặp hoặc sai căn cứ. Giữ nguyên đáp án baseline nếu sửa không đạt.

**Add Input:** submission baseline; diagnostics Stage 2 đúng private; model Version 3;
output Stage 2/3 chứa `selected_adapter/adapter_model.safetensors` của epoch đã chọn.
ZIP diagnostics không chứa trọng số. Không cần index cho adaptive.
`PRIVATE_DIAGNOSTICS` là tùy chọn: chỉ dùng Stage 3 hoàn chỉnh đúng private.
Stage 3 public paused 900/1.000 câu không khớp và không được ghép vào private.

`adaptive_audit` chỉ dùng tokenizer, không load model GPU. `adaptive_dev` chạy smoke tối đa 5 ID
trong phiên đầu, sau đó resume tối đa 50 ID mới/phiên. `adaptive_private` tự đọc kết quả dev
cùng identity và chỉ áp dụng nhóm đạt METEOR không giảm. Không cần chọn winner theo từng câu.
Dev100 đã dùng chọn adapter: kết quả chỉ là screening, không đảm bảo tăng điểm private.

Mode cũ vẫn có: `p1_dev`, `p1_public`, `p2_retrieval`, `p2_generate`, `repair_v2`;
các mode này vẫn cần diagnostics Stage 3 hoàn chỉnh như trước.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess

SESSION_STARTED = time.monotonic()
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Notebook này dùng đường dẫn Kaggle. Chạy local bằng python -m legalqa.repair.')

# Bundle đã đổi: chạy adaptive_dev trong output mới, không resume bundle cũ.
MODE = 'adaptive_dev'  # adaptive_audit | adaptive_dev | adaptive_private | p1_dev | p1_public | p2_retrieval | p2_generate | repair_v2
BASELINE_SUBMISSION = None   # ZIP, submission.json, hoặc thư mục chứa submission.json: bản 0.5713.
STAGE2_DIAGNOSTICS = None    # ZIP Stage 2 hoặc thư mục đã giải nén, đúng private 1.918 câu.
PRIVATE_DIAGNOSTICS = None   # Tùy chọn: Stage 3 hoàn chỉnh đúng private; không tự chọn public cũ.

# None: tự tìm đúng một diagnostics ZIP, hoặc một thư mục Stage 3 đã giải nén.
DIAGNOSTICS = None
EXPECTED_DIAGNOSTICS_SHA256 = None  # New private diagnostics; identity is locked on first run.
DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')
TEST_PATH = DATASET_ROOT / 'private-official.json'
OUTPUT = WORK / 'legalqa_main_04_v8_private_adaptive'
RUN_GPU = True
MODEL_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1/models')
ADAPTER_ROOT = None          # Thư mục selected_adapter chứa trọng số + adapter_config.json.
PREVIOUS_OUTPUT = None       # Chỉ đặt output cùng bundle này khi resume một mode bị paused.
P1_WINNER = None             # Bắt buộc với p1_public, ví dụ 'g1_penalty_103'.
P2_SHORTLIST = []            # p2_generate: tối đa 2 tên từ báo cáo p2_retrieval.
P1_VARIANTS = ['g0_penalty_100', 'g1_penalty_103', 'g1_penalty_105']
P2_VARIANTS = ['r1_pool_64', 'r2_intent_query', 'r3_adjacent_articles',
               'r4_lexical_weight_1', 'r5_scope_penalty']
GPU_MAX_ITEMS = 50           # Số câu mới mỗi variant/process trong phiên này.
INSTALL_DEPS = True          # Tắt nếu môi trường đã có scorer dependencies + WordNet.
AUDIT_ONLY = False           # Chỉ áp dụng cho mode repair_v2.
WORK_HOURS = 9.0             # Gồm cài đặt, CPU, GPU và chấm; không cam kết xong trong một phiên.
VALID_MODES = {'adaptive_audit', 'adaptive_dev', 'adaptive_private',
               'p1_dev', 'p1_public', 'p2_retrieval', 'p2_generate', 'repair_v2'}
if MODE not in VALID_MODES:
    raise ValueError(f'MODE không hợp lệ: {MODE}')
if not 0 < WORK_HOURS <= 9:
    raise ValueError('WORK_HOURS phải nằm trong (0, 9].')
DEADLINE = SESSION_STARTED + WORK_HOURS * 3600
if MODE not in {'repair_v2', 'adaptive_audit'} and not RUN_GPU:
    raise ValueError(f'{MODE} cần RUN_GPU=True.')
if MODE != 'repair_v2' and AUDIT_ONLY:
    raise ValueError('AUDIT_ONLY chỉ dùng với repair_v2.')
if RUN_GPU and AUDIT_ONLY:
    raise ValueError('RUN_GPU không dùng cùng AUDIT_ONLY.')
if not isinstance(GPU_MAX_ITEMS, int) or GPU_MAX_ITEMS <= 0:
    raise ValueError('GPU_MAX_ITEMS phải là số nguyên dương.')
if MODE == 'p1_public' and P1_WINNER is None:
    raise ValueError('p1_public yêu cầu P1_WINNER.')
if MODE == 'p2_generate' and not (1 <= len(P2_SHORTLIST) <= 2):
    raise ValueError('p2_generate yêu cầu P2_SHORTLIST có 1 hoặc 2 variant.')

def run_bounded(command, **kwargs):
    remaining = DEADLINE - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('Hết ngân sách Stage 4; chưa xác nhận kết quả của phiên này.')
    return subprocess.run(list(map(str, command)), check=True, timeout=remaining, **kwargs)

## Code đã đóng gói

Cell sau chứa bản sao code và scorer của notebook này, kèm SHA-256. Không cần sửa payload. Muốn thay đổi thuật toán trong repo, chạy `python scripts/build_stage4_notebook.py` để tạo lại notebook.

In [ ]:
BUNDLE_SHA256 = '526e488accc6607b871532a1d05cd1d9aed3ed4405993343d2e2e47ae774e7f4'
BUNDLE_B64 = '...'

In [ ]:
import base64, hashlib, io, zipfile
from pathlib import PurePosixPath

payload = base64.b64decode(BUNDLE_B64)
if hashlib.sha256(payload).hexdigest() != BUNDLE_SHA256:
    raise ValueError('Payload code không khớp SHA-256.')
CODE = WORK / ('legalqa_stage4_code_' + BUNDLE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        part = PurePosixPath(name)
        if part.is_absolute() or '..' in part.parts or '\\' in name or ':' in name:
            raise ValueError('Đường dẫn không hợp lệ trong code bundle.')
        target = CODE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))

NLTK_ROOT = WORK / 'stage4_nltk_data'
env = dict(os.environ)
env['NLTK_DATA'] = str(NLTK_ROOT) + os.pathsep + env.get('NLTK_DATA', '')
env['PYTHONPATH'] = str(CODE)
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['LEGALQA_DEADLINE'] = str(time.time() + max(0, DEADLINE - time.monotonic()))
env['LEGALQA_MAX_ITEMS'] = '0'
if INSTALL_DEPS and not AUDIT_ONLY:
    run_bounded([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 'numpy>=1.26,<3', 'nltk==3.9.1', 'absl-py==2.2.2', 'six==1.17.0'])
    if RUN_GPU or MODE.startswith('adaptive'):
        # Retain Kaggle CUDA torch. The model contract matches the Stage 3 environment.
        run_bounded([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                     'transformers==4.51.3', 'accelerate==1.6.0', 'peft==0.15.2',
                     'bitsandbytes==0.45.5', 'huggingface-hub==0.30.2',
                     'safetensors==0.5.3', 'sentencepiece==0.2.0'])
    if MODE.startswith('p2'):
        run_bounded([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                     'faiss-cpu==1.10.0', 'ijson==3.4.0.post0'])
    # A failed resource download must stop the run, rather than silently changing METEOR.
    run_bounded([sys.executable, '-c',
        'import nltk; nltk.download("wordnet", download_dir=' + repr(str(NLTK_ROOT)) + ', raise_on_error=True); '
        'nltk.download("omw-1.4", download_dir=' + repr(str(NLTK_ROOT)) + ', raise_on_error=True)'], env=env)
if not AUDIT_ONLY:
    run_bounded([sys.executable, '-c',
                 'from legalqa.metrics import metric_environment; metric_environment(); print("Scorer ready")'],
                cwd=CODE, env=env)
print('Code:', CODE)

## Input và resume

Adaptive tự tìm duy nhất một baseline và một Stage 2; nếu nhiều nguồn, điền đường dẫn trong cell cấu hình.
Kaggle giải nén ZIP cũng được hỗ trợ. Không tự chọn Stage 3.
Để resume, Add Input output phiên trước, đặt `PREVIOUS_OUTPUT` tới thư mục có `main04_state.json`.
Giữ nguyên input/model/code; chuyển `adaptive_dev` → `adaptive_private` dùng cùng output.

In [ ]:
if MODE.startswith('adaptive'):
    sys.path.insert(0, str(CODE))
    from legalqa.adaptive_inputs import load_adaptive_inputs
    def choose_input(value, pattern, marker=None):
        if value is not None:
            path = Path(value)
            if not path.exists():
                raise FileNotFoundError(path)
            return path
        matches = sorted(INPUT.rglob(pattern))
        if not matches and marker:
            matches = sorted(p.parent for p in INPUT.rglob(marker))
        if len(matches) != 1:
            raise ValueError(f'Chọn input cụ thể cho {pattern}: {matches}')
        return matches[0]
    STAGE2_DIAGNOSTICS = choose_input(STAGE2_DIAGNOSTICS,
        'legalqa_main_stage2_v8_diagnostics*.zip', 'stage2_manifest.json')
    BASELINE_SUBMISSION = choose_input(BASELINE_SUBMISSION, 'submission.zip', 'submission.json')
    if PRIVATE_DIAGNOSTICS is not None and Path(PRIVATE_DIAGNOSTICS).is_dir():
        from legalqa.repair import diagnostics_zip_from_directory
        PRIVATE_DIAGNOSTICS = diagnostics_zip_from_directory(
            PRIVATE_DIAGNOSTICS, WORK / 'adaptive_private_diagnostics.zip')
    checked = load_adaptive_inputs(STAGE2_DIAGNOSTICS, BASELINE_SUBMISSION, PRIVATE_DIAGNOSTICS)
    diagnostics_sha256 = checked['source']['stage2']
    print('Verified private IDs:', len(checked['private']['questions']))
    print('Private audit:', 'matched' if checked['private']['audit'] is not None else 'unavailable; heuristic mode')
    print('Baseline score reported by user: 0.5713; Stage 3: 0.5704')
    del checked
else:
    if DIAGNOSTICS is None:
        matches = sorted(INPUT.rglob('legalqa_main_stage3_v8_diagnostics*.zip'))
        if not matches:
            matches = sorted(p.parent for p in INPUT.rglob('stage3_manifest.json'))
        if len(matches) != 1:
            raise RuntimeError(f'Cần đúng một input Stage 3. Tìm thấy {len(matches)}: {matches}. Đặt DIAGNOSTICS cụ thể.')
        DIAGNOSTICS = matches[0]
    DIAGNOSTICS = Path(DIAGNOSTICS)
    if not DIAGNOSTICS.exists():
        raise FileNotFoundError(DIAGNOSTICS)
    diagnostics_was_directory = DIAGNOSTICS.is_dir()
    if diagnostics_was_directory:
        packed = WORK / 'stage4_input_diagnostics.zip'
        run_bounded([sys.executable, '-c',
            'import sys; from legalqa.repair import diagnostics_zip_from_directory; '
            'diagnostics_zip_from_directory(sys.argv[1], sys.argv[2])', DIAGNOSTICS, packed], cwd=CODE, env=env)
        DIAGNOSTICS = packed
    diagnostics_sha256 = hashlib.sha256(DIAGNOSTICS.read_bytes()).hexdigest()
    if (EXPECTED_DIAGNOSTICS_SHA256 and not diagnostics_was_directory
            and diagnostics_sha256 != EXPECTED_DIAGNOSTICS_SHA256):
        raise ValueError(f'Sai diagnostics SHA-256: {diagnostics_sha256}')

    # P2 dùng trực tiếp questions/references/config trong diagnostics. Không tin đường dẫn ZIP.
    EXTRACTED = WORK / ('stage4_diagnostics_' + diagnostics_sha256[:12])
    if not EXTRACTED.exists():
        with zipfile.ZipFile(DIAGNOSTICS) as archive:
            for info in archive.infolist():
                part = PurePosixPath(info.filename)
                if part.is_absolute() or '..' in part.parts or '\\' in info.filename or ':' in info.filename:
                    raise ValueError(f'Đường dẫn diagnostics không hợp lệ: {info.filename}')
            archive.extractall(EXTRACTED)
    # Match all private IDs and question text before any dev/test processing.
    sys.path.insert(0, str(CODE))
    from legalqa.io import load_questions
    if not TEST_PATH.is_file():
        raise FileNotFoundError(TEST_PATH)
    private_questions = load_questions(TEST_PATH)
    if load_questions(EXTRACTED / 'data/test.questions.json') != private_questions:
        raise ValueError('Diagnostics do not match private-official.json. Run Stage 2/3 on private data first.')
    print('Private questions:', len(private_questions), '| Input:', TEST_PATH)
    print('Diagnostics:', DIAGNOSTICS)
    print('Diagnostics SHA-256:', diagnostics_sha256)
    print('Output:', OUTPUT)

In [ ]:
import shutil
if PREVIOUS_OUTPUT is not None and not OUTPUT.exists():
    previous = Path(PREVIOUS_OUTPUT)
    if not (previous / 'main04_state.json').is_file():
        raise ValueError('PREVIOUS_OUTPUT phải là output Main 04 mới có main04_state.json.')
    shutil.copytree(previous, OUTPUT)
if RUN_GPU or MODE.startswith('adaptive'):
    if MODEL_ROOT is None:
        choices = sorted(p.parent for p in INPUT.rglob('models.lock.json')
                         if (p.parent / 'generator/config.json').is_file())
        if len(choices) != 1:
            raise RuntimeError(f'Đặt MODEL_ROOT cụ thể; tìm thấy {choices}.')
        MODEL_ROOT = choices[0]
    if ADAPTER_ROOT is None:
        choices = sorted(p.parent for p in INPUT.rglob('selected_adapter/adapter_config.json')
                         if (p.parent / 'adapter_model.safetensors').is_file())
        if len(choices) != 1:
            raise RuntimeError(f'Đặt ADAPTER_ROOT cụ thể; tìm thấy {choices}.')
        ADAPTER_ROOT = choices[0]
    print('Models:', MODEL_ROOT, 'Adapter:', ADAPTER_ROOT)
    print('GPU sẽ kiểm adapter hash và model lock trước khi load weights.')
if MODE.startswith('p2'):
    INDEX_ROOT = MODEL_ROOT.parent / 'index'
    for required in ('index_manifest.json', 'corpus.sqlite', 'dense.faiss'):
        if not (INDEX_ROOT / required).is_file():
            raise FileNotFoundError(INDEX_ROOT / required)
    print('Index:', INDEX_ROOT)

## Chạy mode đã chọn

- `p1_dev`: tái lập P0 và thử ba mức penalty 1.00/1.03/1.05 trên cùng nhóm câu lặp được phát hiện từ output gốc.
- `p1_public`: yêu cầu `P1_WINNER`; tự kiểm `decision.json`, resume tối đa `GPU_MAX_ITEMS` câu và đóng ZIP khi hoàn tất.
- `p2_retrieval`: tạo năm cache retrieval + diagnostic, chưa generation.
- `p2_generate`: yêu cầu `P2_SHORTLIST` tối đa hai variant; resume generation dev100, repair/chấm/paired comparison khi đủ.
- `repair_v2`: workflow Stage 4 V2 cũ.

Không dùng reference trong prompt hoặc chọn candidate theo từng ID. Mọi cache/journal giữ identity riêng. Khi trạng thái `paused`, Save output, Add Input version đó, đặt `PREVIOUS_OUTPUT`, giữ nguyên code/cấu hình và chạy lại.

In [ ]:
sys.path.insert(0, str(CODE))
from legalqa.experiments import INFERENCE_VARIANTS, RETRIEVAL_VARIANTS
from legalqa.io import read_json, write_json
if P1_VARIANTS != list(INFERENCE_VARIANTS) or P2_VARIANTS != list(RETRIEVAL_VARIANTS):
    raise ValueError('Danh sách variant trong notebook khác code bundle.')

RUN_SUCCEEDED = False
OUTPUT.mkdir(parents=True, exist_ok=True)
STATE_PATH = OUTPUT / 'main04_state.json'
state_identity = {'diagnostics_sha256': diagnostics_sha256, 'bundle_sha256': BUNDLE_SHA256}
if STATE_PATH.is_file():
    state = read_json(STATE_PATH)
    if state.get('identity') != state_identity:
        raise ValueError('PREVIOUS_OUTPUT khác diagnostics/code; dùng output mới.')
else:
    state = {'identity': state_identity, 'runs': {}}

def record(status, **details):
    state['runs'][MODE] = {'status': status, **details}
    state['last_mode'] = MODE
    write_json(STATE_PATH, state)

BASELINE = OUTPUT / 'baseline'
EXPECTED_BASELINE_METEOR = None  # Recompute dev baseline for the new private diagnostics.

def ensure_baseline():
    manifest = BASELINE / 'repair.manifest.json'
    if not manifest.is_file() or read_json(manifest).get('status') != 'complete':
        run_bounded([sys.executable, '-m', 'legalqa.repair_v2',
                     '--diagnostics', DIAGNOSTICS, '--output', BASELINE], cwd=CODE, env=env)
    metrics = read_json(BASELINE / 'dev.selected.metrics.json')
    if EXPECTED_BASELINE_METEOR is not None and abs(metrics['meteor'] - EXPECTED_BASELINE_METEOR) > 1e-10:
        raise ValueError(f'Không tái lập đúng P0: {metrics["meteor"]}')
    return metrics

CONFIG_ROOT = OUTPUT / 'configs'
def ensure_configs():
    manifest = CONFIG_ROOT / 'manifest.json'
    if not manifest.is_file():
        run_bounded([sys.executable, '-m', 'legalqa.experiments', 'write-configs',
                     '--base', EXTRACTED / 'config.json', '--output', CONFIG_ROOT], cwd=CODE, env=env)
    return manifest

if MODE.startswith('adaptive'):
    target = OUTPUT / 'adaptive'
    command = [sys.executable, '-m', 'legalqa.adaptive', '--mode', MODE,
               '--stage2', STAGE2_DIAGNOSTICS, '--submission', BASELINE_SUBMISSION,
               '--models', MODEL_ROOT, '--adapter', ADAPTER_ROOT, '--output', target,
               '--max-items', GPU_MAX_ITEMS]
    if PRIVATE_DIAGNOSTICS is not None:
        command += ['--private-diagnostics', PRIVATE_DIAGNOSTICS]
    run_bounded(command, cwd=CODE, env=env)
    adaptive_status = read_json(target / 'status.json')
    record(adaptive_status['status'], output='adaptive',
           submission_zip=('adaptive/' + adaptive_status['submission_zip'])
           if adaptive_status.get('submission_zip') else None)

elif MODE == 'p1_dev':
    baseline_metrics = ensure_baseline()
    root = OUTPUT / 'p1'
    summary = {}
    for variant in P1_VARIANTS:
        target = root / f'{variant}_dev'
        run_bounded([sys.executable, '-m', 'legalqa.experiments', 'inference',
                     '--diagnostics', DIAGNOSTICS, '--baseline', BASELINE,
                     '--models', MODEL_ROOT, '--adapter', ADAPTER_ROOT,
                     '--variant', variant, '--output', target, '--split', 'dev',
                     '--max-items', GPU_MAX_ITEMS], cwd=CODE, env=env)
        status = read_json(target / 'status.json')
        summary[variant] = status
        decision = target / 'decision.json'
        metrics = target / 'dev.candidate.metrics.json'
        if decision.is_file():
            summary[variant]['decision'] = read_json(decision)
        if metrics.is_file():
            score = read_json(metrics)
            summary[variant]['metrics'] = {key: score[key] for key in ('meteor', 'rougeL')}
        if status.get('status') == 'paused':
            break
    penalty_control = summary.get('g0_penalty_100', {}).get('metrics')
    comparison = {'control_variant': 'g0_penalty_100', 'control_metrics': penalty_control,
                  'variants': {}}
    if penalty_control:
        for variant in ('g1_penalty_103', 'g1_penalty_105'):
            metrics = summary.get(variant, {}).get('metrics')
            if metrics:
                comparison['variants'][variant] = {
                    'metrics': metrics,
                    'delta_vs_1_00': {key: metrics[key] - penalty_control[key]
                                      for key in ('meteor', 'rougeL')},
                }
    write_json(root / 'penalty_comparison.json', comparison)
    write_json(root / 'p1_summary.json', {'baseline': {k: baseline_metrics[k] for k in ('meteor','rougeL')},
                                          'penalty_comparison': comparison, 'variants': summary})
    complete = len(summary) == len(P1_VARIANTS) and all(
        row.get('status') == 'complete' for row in summary.values())
    record('complete' if complete else 'paused', summary='p1/p1_summary.json',
           penalty_comparison='p1/penalty_comparison.json')

elif MODE == 'p1_public':
    ensure_baseline()
    root = OUTPUT / 'p1'
    dev_result = root / f'{P1_WINNER}_dev'
    decision = read_json(dev_result / 'decision.json')
    if not decision.get('passes_screen'):
        raise ValueError(f'{P1_WINNER} không qua điều kiện dev; không chạy public.')
    target = root / f'{P1_WINNER}_public'
    run_bounded([sys.executable, '-m', 'legalqa.experiments', 'inference',
                 '--diagnostics', DIAGNOSTICS, '--baseline', BASELINE,
                 '--models', MODEL_ROOT, '--adapter', ADAPTER_ROOT,
                 '--variant', P1_WINNER, '--dev-result', dev_result,
                 '--output', target, '--split', 'public',
                 '--max-items', GPU_MAX_ITEMS], cwd=CODE, env=env)
    status = read_json(target / 'status.json')
    zip_path = None
    if status.get('status') == 'complete':
        zip_path = OUTPUT / f'submission_{P1_WINNER}.zip'
        run_bounded([sys.executable, '-m', 'legalqa', '--config', EXTRACTED / 'config.json',
                     'package', '--predictions', target / 'public.candidate.json',
                     '--questions', EXTRACTED / 'data/test.questions.json',
                     '--output', zip_path], cwd=CODE, env=env)
    record(status.get('status', 'paused'), winner=P1_WINNER,
           submission_zip=zip_path.name if zip_path else None)

elif MODE == 'p2_retrieval':
    ensure_configs()
    root = OUTPUT / 'p2'
    summary = {}
    for variant in P2_VARIANTS:
        cfg = CONFIG_ROOT / 'retrieval' / f'{variant}.json'
        target = root / variant
        retrieval = target / 'dev100.retrieval.json'
        diagnostic = target / 'retrieval.diagnostic.json'
        run_bounded([sys.executable, '-m', 'legalqa', '--config', cfg, '--models', MODEL_ROOT,
                     'retrieve', '--questions', EXTRACTED / 'data/dev100.questions.json',
                     '--index', INDEX_ROOT, '--output', retrieval], cwd=CODE, env=env)
        if not retrieval.is_file():
            summary[variant] = {'status': 'paused'}
            break
        run_bounded([sys.executable, '-m', 'legalqa', '--config', cfg,
                     'diagnose-retrieval', '--qa', EXTRACTED / 'data/dev100.json',
                     '--retrieval', retrieval, '--index', INDEX_ROOT,
                     '--output', diagnostic], cwd=CODE, env=env)
        report = read_json(diagnostic)
        summary[variant] = {'status': 'complete', 'values': report['values']}
    write_json(root / 'retrieval_summary.json', summary)
    complete = len(summary) == len(P2_VARIANTS) and all(
        row.get('status') == 'complete' for row in summary.values())
    record('complete' if complete else 'paused', summary='p2/retrieval_summary.json')

    # Gói riêng báo cáo nhẹ để tải/chia sẻ, không đưa cache retrieval lớn vào ZIP.
    diagnostic_zip = OUTPUT / 'p2_retrieval_diagnostics.zip'
    diagnostic_tmp = diagnostic_zip.with_suffix('.zip.tmp')
    diagnostic_files = [root / 'retrieval_summary.json', OUTPUT / 'main04_state.json']
    diagnostic_files += [root / variant / 'retrieval.diagnostic.json'
                         for variant in P2_VARIANTS
                         if (root / variant / 'retrieval.diagnostic.json').is_file()]
    with zipfile.ZipFile(diagnostic_tmp, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in diagnostic_files:
            archive.write(path, path.relative_to(OUTPUT).as_posix())
    os.replace(diagnostic_tmp, diagnostic_zip)

elif MODE == 'p2_generate':
    ensure_configs()
    baseline_metrics = ensure_baseline()
    unknown = sorted(set(P2_SHORTLIST) - set(P2_VARIANTS))
    if unknown:
        raise ValueError(f'P2_SHORTLIST không hợp lệ: {unknown}')
    root = OUTPUT / 'p2'
    summary = {}
    generation_env = {**env, 'LEGALQA_MAX_ITEMS': str(GPU_MAX_ITEMS)}
    for variant in P2_SHORTLIST:
        cfg = CONFIG_ROOT / 'retrieval' / f'{variant}.json'
        target = root / variant
        retrieval = target / 'dev100.retrieval.json'
        if not retrieval.is_file():
            raise FileNotFoundError(f'Chạy p2_retrieval trước: {retrieval}')
        raw = target / 'dev.raw.json'
        run_bounded([sys.executable, '-m', 'legalqa', '--config', cfg, '--models', MODEL_ROOT,
                     'generate', '--questions', EXTRACTED / 'data/dev100.questions.json',
                     '--retrieval', retrieval, '--adapter', ADAPTER_ROOT,
                     '--output', raw], cwd=CODE, env=generation_env)
        if not raw.is_file():
            partial = raw.with_suffix('.partial.json')
            summary[variant] = {'status': 'paused',
                                'generated': len(read_json(partial)) if partial.is_file() else 0}
            continue
        repaired = target / 'dev.repaired.json'
        run_bounded([sys.executable, '-m', 'legalqa.experiments', 'postprocess',
                     '--predictions', raw, '--audit', raw.with_suffix('.audit.json'),
                     '--output', repaired], cwd=CODE, env=env)
        base_report = target / 'baseline.metrics.json'
        candidate_report = target / 'dev.metrics.json'
        paired = target / 'paired.json'
        run_bounded([sys.executable, '-m', 'legalqa', '--config', cfg, 'evaluate',
                     '--predictions', BASELINE / 'dev.selected.json',
                     '--references', EXTRACTED / 'data/dev100.references.json',
                     '--output', base_report, '--label', 'baseline_repaired'], cwd=CODE, env=env)
        run_bounded([sys.executable, '-m', 'legalqa', '--config', cfg, 'evaluate',
                     '--predictions', repaired,
                     '--references', EXTRACTED / 'data/dev100.references.json',
                     '--output', candidate_report, '--label', variant], cwd=CODE, env=env)
        run_bounded([sys.executable, '-m', 'legalqa', '--config', cfg, 'compare',
                     '--baseline', base_report, '--candidate', candidate_report,
                     '--output', paired], cwd=CODE, env=env)
        scores = read_json(candidate_report)
        summary[variant] = {'status': 'complete', 'meteor': scores['meteor'],
                            'rougeL': scores['rougeL'], 'paired': read_json(paired)}
    write_json(root / 'generation_summary.json',
               {'baseline': {k: baseline_metrics[k] for k in ('meteor','rougeL')},
                'variants': summary})
    complete = len(summary) == len(P2_SHORTLIST) and all(
        row.get('status') == 'complete' for row in summary.values())
    record('complete' if complete else 'paused', summary='p2/generation_summary.json')

else:  # repair_v2 compatibility mode
    target = OUTPUT / 'repair_v2'
    command = [sys.executable, '-m', 'legalqa.repair_v2',
               '--diagnostics', DIAGNOSTICS, '--output', target]
    if AUDIT_ONLY:
        command.append('--audit-only')
    if RUN_GPU:
        command.extend(['--gpu', '--models', MODEL_ROOT, '--adapter', ADAPTER_ROOT,
                        '--max-items', GPU_MAX_ITEMS])
    run_bounded(command, cwd=CODE, env=env)
    manifest = read_json(target / 'repair.manifest.json')
    record(manifest.get('status', 'paused'), output='repair_v2',
           submission_zip=manifest.get('submission_zip'))

RUN_SUCCEEDED = True

In [ ]:
if not globals().get('RUN_SUCCEEDED', False):
    raise RuntimeError('Chưa có lần chạy Stage 4 thành công trong phiên này.')
from IPython.display import display, FileLink
state = read_json(OUTPUT / 'main04_state.json')
current = state['runs'][MODE]
print('MODE:', MODE, '| STATUS:', current['status'])
print(json.dumps(current, ensure_ascii=False, indent=2))

links = [OUTPUT / 'main04_state.json']
if MODE.startswith('adaptive'):
    links += [OUTPUT / 'adaptive/status.json', OUTPUT / 'adaptive/audit.json',
              OUTPUT / 'adaptive/dev.status.json', OUTPUT / 'adaptive/private.status.json']
    for method in ('audit', 'heuristic'):
        links += [OUTPUT / f'adaptive/dev/{method}/decision.json',
                  OUTPUT / f'adaptive/private/{method}/outcomes.json',
                  OUTPUT / f'adaptive/private/{method}/unresolved.json',
                  OUTPUT / f'adaptive/private/{method}/skipped.json']
    if current.get('submission_zip'):
        links.append(OUTPUT / current['submission_zip'])
elif MODE == 'p1_dev':
    links += [OUTPUT / 'p1/p1_summary.json', OUTPUT / 'p1/penalty_comparison.json']
elif MODE == 'p1_public':
    links += [OUTPUT / f'p1/{P1_WINNER}_public/status.json']
    if current.get('submission_zip'):
        links.append(OUTPUT / current['submission_zip'])
elif MODE == 'p2_retrieval':
    links += [OUTPUT / 'p2/retrieval_summary.json',
              OUTPUT / 'p2_retrieval_diagnostics.zip']
elif MODE == 'p2_generate':
    links.append(OUTPUT / 'p2/generation_summary.json')
else:
    links += [OUTPUT / 'repair_v2/repair.metrics.json',
              OUTPUT / 'repair_v2/repair.manifest.json']
    if current.get('submission_zip'):
        links.append(OUTPUT / 'repair_v2' / current['submission_zip'])

for path in links:
    if path.is_file():
        display(FileLink(str(path)))
if current['status'] == 'paused':
    print('Save toàn bộ output, Add Input version này, đặt PREVIOUS_OUTPUT rồi chạy lại cùng MODE/config.')
print('Điểm P1/P2 hiện tại là dev100; chưa phải bằng chứng private >= 0.60.')

## Đọc kết quả adaptive

Khi `paused`, Save output và resume cùng mode. Phiên đầu smoke 5 ID, các phiên sau tối đa
`GPU_MAX_ITEMS=50` ID mới trong tổng `WORK_HOURS=9` (gồm setup, mọi retry và chấm).
Xem `adaptive/dev/{audit,heuristic}/decision.json`: mỗi nhóm và bản ghép phải đạt METEOR
không giảm, có thay đổi được chấp nhận và không tăng câu lặp nặng/dang dở.
Sau khi `adaptive_dev` complete, đổi `MODE='adaptive_private'`; không chỉnh policy hoặc input.

`adaptive/submission_adaptive.zip` chỉ được xuất khi private hoàn tất, chỉ chứa `submission.json`.
Nếu không có nhóm dev đạt điều kiện, ZIP giữ baseline. Xem `outcomes.json`, `unresolved.json`
và `skipped.json` để biết câu nào đã thay, thất bại hoặc bị loại bởi dev.
Giữ baseline 0.5713 để đối chiếu; chỉ lần nộp tiếp theo mới xác nhận điểm private mới.